In [ ]:
# Load mean runtimes from the standardized benchmark results.
# Regenerate with:  python bench/run.py matrix   &&   python bench/aggregate.py
import pandas as pd
from pathlib import Path

SUMMARY = Path("../../results/summary.csv")
_df = pd.read_csv(SUMMARY)

_LANG_ORDER = ["python", "go", "cpp", "csharp", "java", "rust"]

def mean_runtimes(algo, model, impl="par", stat="mean_ms"):
    """Mean runtime (ms) for one model's parallel run of `algo`, ordered as
    [Python, Go, C++, C#, Java, Rust]. Missing combos come back as NaN."""
    sub = _df[(_df.algo == algo) & (_df.model == model) & (_df.impl == impl)]
    by_lang = sub.set_index("lang")[stat].to_dict()
    return [by_lang.get(l, float("nan")) for l in _LANG_ORDER]

# Example (bfs); swap algo/model as needed for each figure:
gpt5 = mean_runtimes("bfs", "gpt-5")
gemini = mean_runtimes("bfs", "gemini-2-5-pro")
claude = mean_runtimes("bfs", "claude-sonnet-4-5")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Languages / groups on x-axis
languages = ["Python", "Go", "C++", "C#", "Java", "Rust"]
x = np.arange(len(languages))

# gpt5 / gemini / claude come from the cell above (mean_runtimes from
# results/summary.csv). If you have not run the benchmarks yet, uncomment
# these fallbacks:
# gpt5 = [1850.58, 53.18, 13.04, 28.71, 44.68, 10.95]
# gemini = [23456.23, 29.76, 31.16, 29.74, 36.30, 15.11]
# claude = [23326.51, 43.22, 20.61, 25.16, 34.65, 519.04]

width = 0.22  # bar width

fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(x - width, gpt5, width, label="GPT-5 Parallel")
ax.bar(x,        claude, width, label="Claude Sonnet 4.5 Parallel")
ax.bar(x + width, gemini, width, label="Gemini 2.5 Pro Parallel")

ax.set_yscale("log")

ax.set_xlabel("Language")
ax.set_ylabel("Mean runtime (ms, log scale)")
ax.set_title("Parallel runtime by model and language")
ax.set_xticks(x)
ax.set_xticklabels(languages)
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)

fig.tight_layout()
plt.show()
